In [1]:
import odds_datasets
from iforest import BAD_IForest
import batch_bald
import numpy as np

for dataset in odds_datasets.datasets_names:
    X, y = odds_datasets.load(dataset)
    model = BAD_IForest(contamination=y.mean()).fit(X)
    print(f"\nDATASET:     {dataset[:10]}\tX: {X.shape}\tcontamination: {100*y.mean():.2f}%")
    for batch_size in [10]:
        print(f"\nBATCH SIZE: {batch_size}")

        # RANDOM
        print("random          \t", end="")
        %timeit -n 10 model.decision_function(X)

        # INDEPENDENT 
        for interest_method in ["bald", "margin", "anom"]:
            model.interest_method = interest_method # type: ignore
            print(f"independent {interest_method}\t", end="")
            %timeit -n 10 model.get_queries(X, batch_size, independent=True) 

        # WORSTCASE
        for interest_method in ["bald", "margin", "anom"]:
            model.interest_method = interest_method # type: ignore
            print(f"worstcase {interest_method}  \t", end="")
            %timeit -n 10 model.get_queries(X, batch_size, independent=False)

        # BATCHBALD
        print(f"batchbald       \t", end="")
        %timeit -n 10 batch_bald.get_queries(model=model, X=X, batch_size=batch_size)


DATASET:     wine	X: (129, 13)	contamination: 7.75%

BATCH SIZE: 10
random          	2.01 ms ± 24.8 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
independent bald	2.04 ms ± 23.1 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
independent margin	2.03 ms ± 20 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
independent anom	2.03 ms ± 24.2 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
worstcase bald  	3.24 ms ± 34.8 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
worstcase margin  	3.22 ms ± 24.8 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
worstcase anom  	2.93 ms ± 26.5 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
batchbald       	35.3 ms ± 277 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)

DATASET:     vertebral	X: (240, 6)	contamination: 12.50%

BATCH SIZE: 10
random          	2.43 ms ± 26.7 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
independent bald	2.45 ms ± 19.3 μs per loop (mean ± std. dev. of